### Libraries.

In [1]:
from functools import partial
import math
import os
import os
import sys
from typing import Literal, OrderedDict

import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import seaborn as sns
import scvi
from sklearn.model_selection import train_test_split
import torch
import torchdiffeq
import torchsde
from tqdm import tqdm

from sc_exp_design.data import DataManager, SequentialDataLoader
from sc_exp_design.utils import set_reproducibility

sys.path.insert(0, "../scripts")
from fetch_data import get_log_transformed_experimental_variables
from data_utils import (
    get_one_hot_encoded_protocols,
    get_one_hot_encoded_protocol_axis
)


### Constants.

In [2]:
# paths
H5AD_PATH = "/Users/lorenzo.consoli/Work/data/collab-goettgens/ds_HID01_logicle_100k_UMAP_ann.h5ad"

# columns
SCATTER_COLUMNS = ['FSC - Area', 'FSC - Height', 'FSC - Width', 'SSC - Area', 'SSC - Height', 'SSC - Width']

# adata keys
X_PCA_OBSM_KEY = "X_pca" 
X_CHANNEL_OBSM_KEY = "X_channel"
X_SCATTER_OBSM_KEY = "X_scatter"
OHE_PROTOCOLS_OBS_KEY = "protocol"
OHE_PROTOCOLS_UNS_KEY = "protocol_ohe"
CONCAT_FEATS_OBSM_KEY_SUFFIX = "scatter_concat"
CONDITION_CONCAT_OBSM_KEY = "protocol_axes_concat"
TARGET_OBS_KEY = "cell_type"
STANDARDIZED_KEY_SUFFIX = "standardized"

# transformations options
STANDARDIZE_SCATTER_FEATURES = True
STANDARDIZE_CHANNEL_FEATURES = True
STANDARDIZE_PCs = False

# training
TRAIN_BATCH_SIZE = 1024
VAL_BATCH_SIZE = 64
LR = 1e-3
NUM_GRAD_STEPS = 1_000
GRAD_STEPS_LOG_INTERVAL = 10

# model
STATE_ENCODER_HIDDEN_DIMS = (128, 128)
DEVICE = "cpu"

# reproducibility
SEED = 42
set_reproducibility(SEED)


### Reading data.

In [3]:
adata = sc.read_h5ad(H5AD_PATH)
adata

AnnData object with n_obs × n_vars = 100000 × 21
    obs: 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'cell_counts', 'experiment_number', 'experiment_id', '740-yp_[µm]', 'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]', 'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC - Area', 'FSC - Height', 'FSC - Width', 'SSC - Area', 'SSC - Height', 'SSC - Width', 'Time Stamp', 'batch', 'leiden', 'cell_type'
    var: 'n', 'channel', 'marker', '$PnD', '$PnB', '$PnR', '$PnG', '$PnE', 'signal_type', 'antibody'
    uns: 'cell_type_colors', 'leiden', 'leiden_colors', 'meta', 'neighbors', 'processing_steps', 'umap'
    obsm: 'X_pca', 'X_umap'
    layers: 'positives', 'raw'
    

### Splitting data.

In [4]:
train_adata, val_adata = train_test_split(adata, test_size=0.3)
train_adata.shape, val_adata.shape

((70000, 21), (30000, 21))

### Tranforming data.

In [5]:
# retrieving scatter features and optionally normalizing them
X_scatter_train = train_adata.obs[SCATTER_COLUMNS].values
X_scatter_val =  val_adata.obs[SCATTER_COLUMNS].values

X_scatter_mean = X_scatter_train.mean(0)
X_scatter_std = X_scatter_train.std(0)

X_scatter_train_std = (X_scatter_train - X_scatter_mean)/X_scatter_std
X_scatter_val_std = (X_scatter_val - X_scatter_mean)/X_scatter_std

assert np.isfinite(X_scatter_train).all()
assert np.isfinite(X_scatter_val).all()

train_adata.obsm[X_SCATTER_OBSM_KEY] = X_scatter_train
train_adata.obsm[F"{X_SCATTER_OBSM_KEY}_{STANDARDIZED_KEY_SUFFIX}"] = X_scatter_train_std

val_adata.obsm[X_SCATTER_OBSM_KEY] = X_scatter_val
val_adata.obsm[F"{X_SCATTER_OBSM_KEY}_{STANDARDIZED_KEY_SUFFIX}"] = X_scatter_val_std


# channel features
train_adata.obsm[X_CHANNEL_OBSM_KEY] = train_adata.X
val_adata.obsm[X_CHANNEL_OBSM_KEY] = val_adata.X

X_channel_train = train_adata.X
X_channel_val = val_adata.X

X_channel_mean = X_channel_train.mean(0)
X_channel_std = X_channel_train.std(0)

X_channel_train_std = (X_channel_train - X_channel_mean)/X_channel_std
X_channel_val_std = (X_channel_val - X_channel_mean)/X_channel_std

assert np.isfinite(X_channel_train).all()
assert np.isfinite(X_channel_val).all()

train_adata.obsm[F"{X_CHANNEL_OBSM_KEY}"] = X_channel_train
train_adata.obsm[F"{X_CHANNEL_OBSM_KEY}_{STANDARDIZED_KEY_SUFFIX}"] = X_channel_train_std

val_adata.obsm[F"{X_CHANNEL_OBSM_KEY}"] = X_channel_val
val_adata.obsm[F"{X_CHANNEL_OBSM_KEY}_{STANDARDIZED_KEY_SUFFIX}"] = X_channel_val_std


# PCs
X_pca_train = train_adata.obsm[X_PCA_OBSM_KEY]
X_pca_val = val_adata.obsm[X_PCA_OBSM_KEY]

X_pca_mean = X_pca_train.mean(0)
X_pca_std = X_pca_train.std(0)

X_pca_train_std = (X_pca_train - X_pca_mean)/X_pca_std
X_pca_val_std = (X_pca_val - X_pca_mean)/X_pca_std

assert np.isfinite(X_pca_train).all()
assert np.isfinite(X_pca_val).all()

train_adata.obsm[f"{X_PCA_OBSM_KEY}"] = X_pca_train
train_adata.obsm[f"{X_PCA_OBSM_KEY}_{STANDARDIZED_KEY_SUFFIX}"] = X_pca_train_std

val_adata.obsm[f"{X_PCA_OBSM_KEY}"] = X_pca_val
val_adata.obsm[f"{X_PCA_OBSM_KEY}_{STANDARDIZED_KEY_SUFFIX}"] = X_pca_val_std

train_adata, val_adata

/var/folders/p4/lyzkxmgs697cq8dyh_h0yh0c0000gn/T/ipykernel_66279/1431344195.py:14: ImplicitModificationWarning: Setting element `.obsm['X_scatter']` of view, initializing view as actual.
  train_adata.obsm[X_SCATTER_OBSM_KEY] = X_scatter_train
/var/folders/p4/lyzkxmgs697cq8dyh_h0yh0c0000gn/T/ipykernel_66279/1431344195.py:17: ImplicitModificationWarning: Setting element `.obsm['X_scatter']` of view, initializing view as actual.
  val_adata.obsm[X_SCATTER_OBSM_KEY] = X_scatter_val


(AnnData object with n_obs × n_vars = 70000 × 21
     obs: 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'cell_counts', 'experiment_number', 'experiment_id', '740-yp_[µm]', 'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]', 'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC - Area', 'FSC - Height', 'FSC - Width', 'SSC - Area', 'SSC - Height', 'SSC - Width', 'Time Stamp', 'batch', 'leiden', 'cell_type'
     var: 'n', 'channel', 'marker', '$PnD', '$PnB', '$PnR', '$PnG', '$PnE', 'signal_type', 'antibody'
     uns: 'cell_type_colors', 'leiden', 'leiden_colors', 'meta', 'neighbors', 'processing_steps', 'umap'
     obsm: 'X_pca', 'X_umap', 'X_scatter', 'X_scatter_standa

### Defining data managers.

In [6]:
# initializing data managers
dm_dict = {}
reprs = [repr for repr in train_adata.obsm_keys() if repr.startswith("X")]
train_data_dict = {}
val_data_dict = {}
for repr in reprs:
    dm = DataManager(
        train_adata,
        None if repr == "X_channel" else repr,
        load_target_covariates=True,
        target_covariates={"cell_type": "one_hot"},
    )
    dm_dict[repr] = dm
    train_data_dict[repr] = dm.get_data(train_adata) 
    val_data_dict[repr] = dm.get_data(val_adata) 
dm_dict.keys()

dict_keys(['X_pca', 'X_umap', 'X_scatter', 'X_scatter_standardized', 'X_channel', 'X_channel_standardized', 'X_pca_standardized'])

### Retrieving chosen representations.

In [7]:
repr = "X_pca"
train_data = train_data_dict[repr]
val_data = val_data_dict[repr]

X_train_original = train_data.state_data
X_train_np = np.concatenate((X_train_original, np.ones((X_train_original.shape[0], 1))), axis=-1)
Y_train_np = train_data.target_data["cell_type"]
X_train = torch.from_numpy(X_train_np).type(torch.float32).to(DEVICE)
Y_train = torch.from_numpy(Y_train_np).type(torch.float32).to(DEVICE)

X_val_original = val_data.state_data
X_val_np = np.concatenate((X_val_original, np.ones((X_val_original.shape[0], 1))), axis=-1)
Y_val_np = val_data.target_data["cell_type"]
X_val = torch.from_numpy(X_val_np).type(torch.float32).to(DEVICE)
Y_val = torch.from_numpy(Y_val_np).type(torch.float32).to(DEVICE)

D = X_train.shape[-1]
K = Y_train.shape[-1]

N_train = X_train.shape[0]
N_val = X_val.shape[0]

print(f"{X_train.shape=}, {Y_train.shape=}")
print(f"{X_val.shape=}, {Y_val.shape=}")

X_train.shape=torch.Size([70000, 21]), Y_train.shape=torch.Size([70000, 19])
X_val.shape=torch.Size([30000, 21]), Y_val.shape=torch.Size([30000, 19])


### Function for Log Likelihood and Score.

In [8]:
def compute_log_likelihood(
    Y, # B, K
    X, # B, D + 1
    W # N, D + 1, K
): # N,
    D = X.shape[-1]
    K = Y.shape[-1]
    N = W.shape[0]

    # check shapes, 
    assert Y.shape[:-1] == X.shape[:-1]

    W = W.reshape(N, D, K)

    assert D == W.shape[1]
    assert K == W.shape[-1]

    # computing logits, probs and ll
    logits = X @ W
    logp = torch.log_softmax(logits, dim=-1)
    logpy = (Y * logp).sum(-1).sum(-1)
    
    assert logits.shape == (N, *X.shape[:-1], K), logits.shape
    assert logpy.shape == (W.shape[0], ), logpy.shape
    return logpy

def compute_grad_log_likelihood(
        Y, # B, K
        X, # B, D + 1
        W, # N, D + 1, K,
    ): # N, (D + 1)*K

    D = X.shape[-1]
    K = Y.shape[-1]
    N = W.shape[0]

    W = W.reshape(N, D, K)

    def _log_likelihood(W):
        return compute_log_likelihood(Y, X, W)
    grads = torch.autograd.functional.vjp(_log_likelihood, W, torch.ones((W.shape[0],), device=W.device, dtype=torch.float32))[1]
    assert grads.shape == (N, D, K), grads.shape
    grads = grads.reshape(N, -1)
    return grads

def compute_analytic_grad_loglikelihood(
    Y, # B, K
    X, # B, D + 1
    W, # N, D + 1, K,
):
    
    D = X.shape[-1]
    K = Y.shape[-1]
    N = W.shape[0]
    W = W.reshape(N, D, K)
    probs = torch.softmax(torch.einsum("bd,ndk->nbk", X, W), dim=-1)
    return torch.einsum("db,nbk->ndk",X.T, (Y.unsqueeze(0) - probs)).reshape(N, -1)


### Test.

In [10]:
N = 15
B = 16
# dummy data
Y = torch.zeros((B, K), device=DEVICE)
Y[:, 0] = 1.0
Y.requires_grad_(True)
X = torch.randn((B, D), requires_grad=True, device=DEVICE, dtype=torch.float32)
W = torch.randn((N, D, K), requires_grad=True, device=DEVICE, dtype=torch.float32)

# test ll
ll = compute_log_likelihood(Y, X, W)
assert ll.shape == (N, )
assert ll.requires_grad

# test grad ll
score = compute_grad_log_likelihood(Y, X, W)
assert score.shape == (N, D *K)

# test flattened ll
ll = compute_log_likelihood(Y, X, W.reshape(N, -1))
assert ll.shape == (N, )
assert ll.requires_grad

# test grad flattened ll
score = compute_grad_log_likelihood(Y, X, W.reshape(N, -1))
assert score.shape == (N, D*K)

# test grad flattened ll
ascore = compute_analytic_grad_loglikelihood(Y, X, W.reshape(N, -1))
assert ascore.shape == (N, D*K)
assert torch.isclose(score, ascore, atol=1e-7).all(), score - ascore
grad_log_likelihood_fn = compute_analytic_grad_loglikelihood

### Function for Log Prior and Score.

In [11]:
def get_sigma_prior(
        X # B, D
    ): # D, D
    B, D = X.shape
    cov = (X.T @ X)
    Icov = torch.linalg.inv(cov)
    assert cov.shape == (D, D)
    assert Icov.shape == (D, D)
    return B*np.pi**2 / (3*(D + 1)) * Icov

def compute_log_prior(
    W, # N, D + 1, K,
    Isigma_prior_reshaped=None, # D, D
    log_det_sigma_prior=None, # scalar 
):
    DK, DK = Isigma_prior_reshaped.shape
    N = W.shape[0]
    if len(W.shape) == 2:
        assert W.shape[1] == DK, f"{W.shape=}, {DK}"
    else:
        N, D, K = W.shape
        assert D*K == DK, f"{D=} {K=} {DK=}"
        W = W.reshape(N, -1)
    
    log_Z = 0.5*(W.shape[-1] * math.log(2 * np.pi) + log_det_sigma_prior)
    potential = torch.einsum("nd, dd->nd", W, Isigma_prior_reshaped)
    potential = -0.5*torch.einsum("nd, nd->n", potential, W)
    
    return potential - log_Z

def compute_grad_log_prior(
    W, # N, D + 1, K,
    Isigma_prior_reshaped=None, # D, D
    log_det_sigma_prior=None, # scalar
):
    DK, DK = Isigma_prior_reshaped.shape
    N = W.shape[0]
    if len(W.shape) == 2:
        K = W.shape[1] // D
    else:
        N, _, K = W.shape
    W = W.reshape(N, D, K)
    def _log_prior(W):
        return compute_log_prior(W, Isigma_prior_reshaped, log_det_sigma_prior)
    grads = torch.autograd.functional.vjp(_log_prior, W, torch.ones((W.shape[0], ), device=W.device, dtype=torch.float32))[1]
    assert grads.shape == (N, D, K), f"{grads.shape}, {(N, D, K)}"
    grads = grads.reshape(N, -1)
    return grads

def compute_analytic_grad_log_prior(
    W, # N, D + 1, K,
    Isigma_prior_reshaped=None, # D, D
):
    DK, DK = Isigma_prior_reshaped.shape
    N = W.shape[0]
    if len(W.shape) == 2:
        assert W.shape[1] == DK, f"{W.shape=}, {DK}"
    else:
        N, D, K = W.shape
        assert D*K == DK, f"{D=} {K=} {DK=}"
        W = W.reshape(N, -1)

    assert Isigma_prior_reshaped.shape == (D*K, D*K), Isigma_prior_reshaped.shape
    return -torch.einsum("nd,dd->nd", W, Isigma_prior_reshaped)

def sample_log_prior(
    N, D, K, sigma_prior_reshaped=None,
):
    assert sigma_prior_reshaped.shape == (D*K, D*K), sigma_prior_reshaped.shape
    noise = torch.randn((N, D*K), device=DEVICE, requires_grad=True, dtype=torch.float32)
    sample = noise @ sigma_prior_reshaped
    return sample

### Test.

In [12]:
# computing inverse covariance and determinant
sigma_prior = get_sigma_prior(X_train)
Isigma_prior = torch.linalg.inv(sigma_prior)
log_det_sigma_prior = torch.log(torch.linalg.det(sigma_prior))
sigma_prior_reshaped = sigma_prior.repeat(K, K)
Isigma_prior_reshaped = Isigma_prior.repeat(K, K)

# attaching to the function
log_prior_fn = partial(compute_log_prior, Isigma_prior_reshaped=Isigma_prior_reshaped, log_det_sigma_prior=log_det_sigma_prior) 
grad_log_prior_fn = partial(compute_grad_log_prior, Isigma_prior_reshaped=Isigma_prior_reshaped, log_det_sigma_prior=log_det_sigma_prior)
analytic_grad_log_prior_fn = partial(compute_analytic_grad_log_prior, Isigma_prior_reshaped=Isigma_prior_reshaped)
sample_prior_fn = partial(sample_log_prior, sigma_prior_reshaped=sigma_prior_reshaped)

# dummy test
lp = log_prior_fn(W)
assert lp.shape == (N, ), f"{lp.shape}, {(N, )}"
assert lp.requires_grad

scorep = grad_log_prior_fn(W)
assert scorep.shape == (N, D *K), scorep.shape

ascorep = analytic_grad_log_prior_fn(W)
assert ascorep.shape == (N, D*K)
assert torch.isclose(scorep, ascorep).all(), scorep - ascorep
grad_log_prior_fn = analytic_grad_log_prior_fn

sigma_prior_sample = sample_log_prior(N, D, K, sigma_prior_reshaped)
assert scorep.shape == (N, D *K), scorep.shape

### Function for Log Posterior and Score.


In [13]:
def compute_log_posterior(
    Y, # B, K
    X, # B, D + 1
    W, # N, D + 1, K
    Isigma_prior_reshaped=Isigma_prior_reshaped, # D, D
    log_det_sigma_prior=log_det_sigma_prior, # scalar
):
    log_p = compute_log_prior(W, Isigma_prior_reshaped, log_det_sigma_prior)
    log_ll = compute_log_likelihood(Y, X, W)
    print(log_p.shape, log_ll.shape)
    return log_p + log_ll

def compute_grad_log_posterior(
    Y, # B, K
    X, # B, D + 1
    W, # N, D + 1, K
    Isigma_prior_reshaped=Isigma_prior_reshaped, # D, D
    log_det_sigma_prior=log_det_sigma_prior, # scalar
):
    glog_p = compute_grad_log_prior(W, Isigma_prior_reshaped, log_det_sigma_prior)
    glog_ll = compute_grad_log_likelihood(Y, X, W)
    return glog_p + glog_ll

def compute_analytic_grad_log_posterior(
    Y, # B, K
    X, # B, D + 1
    W, # N, D + 1, K
    Isigma_prior_reshaped=Isigma_prior_reshaped, # D, D
    log_det_sigma_prior=log_det_sigma_prior, # scalar
):
    glog_p = compute_analytic_grad_log_prior(W, Isigma_prior_reshaped)
    glog_ll = compute_analytic_grad_loglikelihood(Y, X, W)
    return glog_p + glog_ll


### Test.

In [14]:
# attaching to the function
log_posterior_fn = partial(compute_log_posterior, Isigma_prior_reshaped=Isigma_prior_reshaped, log_det_sigma_prior=log_det_sigma_prior) 
grad_log_posterior_fn = partial(compute_grad_log_posterior, Isigma_prior_reshaped=Isigma_prior_reshaped, log_det_sigma_prior=log_det_sigma_prior) 
analytic_grad_log_posterior_fn = partial(compute_analytic_grad_log_posterior, Isigma_prior_reshaped=Isigma_prior_reshaped, log_det_sigma_prior=log_det_sigma_prior) 

lps = log_posterior_fn(Y, X, W)
assert lps.shape == (N, ), lps.shape
assert lps.requires_grad

glps = grad_log_posterior_fn(Y, X, W)
assert glps.shape == (N, D * K)

aglps = analytic_grad_log_posterior_fn(Y, X, W)
assert aglps.shape == (N, D * K)
assert torch.isclose(aglps, glps, atol=1e-7).all(), aglps - glps


torch.Size([15]) torch.Size([15])


### Defining class for handling SDEs.

In [ ]:
class MySDE(torch.nn.Module):
    def __init__(
        self,
        X,
        Y,
        sde_type="ito",
        noise_type="diagonal",
    ):
        super().__init__()
        self.X = X
        self.Y = Y
        self.sde_type = sde_type
        self.noise_type = noise_type

    def f(self, t, W):
        return grad_log_posterior_fn(self.Y, self.X, W)
    
    def g(self, t, W):
        return torch.sqrt(2* torch.ones_like(W))
T = 2
N = 1_000
sde = MySDE(X_train, Y_train)
prior_samples = sample_log_prior(N, D, K, sigma_prior_reshaped)
prior_samples.shape

langevin_samples = torchsde.sdeint(sde, y0=prior_samples, ts=torch.linspace(0.0, 1.0, T), method="euler")
print(langevin_samples.shape)

### 